<a href="https://colab.research.google.com/github/appolinecontact-gif/AI-character/blob/main/LoRA%20data%20inspectionV_2_Unstble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ LoRA Metadata Updated Inspector

A professional Python utility to scan directories for `.safetensors` files and extract crucial metadata like **Trigger Words** and **Base Model** info.

## ✨ Features
* **Auto-Discovery**: Recursively scans all subfolders for LoRA files.
* **Smart Extraction**: Uses a 3-tier priority system (Trained Words > ModelSpec > Tag Frequency) to find the activation token.
* **Clean UI**: Outputs a structured table directly in your terminal.

In [ ]:
import os
import glob
import json
from safetensors.torch import safe_open

def extract_metadata(file_path):
    try:
        with safe_open(file_path, framework="pt") as f:
            metadata = f.metadata()
            if not metadata:
                return None

            # 1. Try to find the explicit trained words (Kohya style)
            trigger = metadata.get("ss_trained_words")

            # 2. If empty, check the ModelSpec title (what you found)
            if not trigger:
                trigger = metadata.get("modelspec.title")

            # 3. If still empty, grab the most frequent tag from the frequency list
            if not trigger:
                tags_json = metadata.get("ss_tag_frequency", "{}")
                try:
                    tags_data = json.loads(tags_json)
                    # Get the very first tag from the first sub-folder/bucket
                    if tags_data:
                        first_bucket = list(tags_data.values())[0]
                        trigger = list(first_bucket.keys())[0] + " (found in tags)"
                except:
                    pass

            return {
                "filename": os.path.basename(file_path),
                "trigger_words": trigger or "Unknown",
                "base_model": metadata.get("ss_sd_model_name", "Unknown"),
                "specification": "ModelSpec" if "modelspec.title" in metadata else "Standard"
            }
    except Exception as e:
        return {"filename": os.path.basename(file_path), "error": str(e)}

def run_scan():
    # Automatically scans the /content/ directory for any safetensors
    files = glob.glob("/content/**/*.safetensors", recursive=True)
    print(f"🔎 Found {len(files)} LoRA files.\n")

    for f in files:
        data = extract_metadata(f)
        if data:
            print(f"📦 {data['filename']}")
            print(f"   🎯 Trigger Word: {data['trigger_words']}")
            print(f"   🛠️  Format: {data['specification']}")
            print("-" * 30)

if __name__ == "__main__":
    run_scan()

# **URL** LoRA inspector

In [2]:
import os
import glob
import json
import struct
import requests
from safetensors.torch import safe_open

class LoraOmniInspector:
    def __init__(self):
        self.headers = {"User-Agent": "Mozilla/5.0"}

    def get_trigger_word(self, metadata):
        """Extracts activation token with fallback logic."""
        return (metadata.get("ss_trained_words") or
                metadata.get("modelspec.title") or
                self._get_from_tags(metadata.get("ss_tag_frequency")))

    def _get_from_tags(self, tags_json):
        if not tags_json:
            return None
            try:
                data = json.loads(tags_json)
                if not data:
                  return None
                first_bucket = list(data.values())[0]
                if not first_bucket:
                  return None
        return list(first_bucket.keys())[0] + " (from tags)"
        except Exception as e:
          print(f"Warning: could not parse tag frequency: {e}")
          return None

    def parse_remote_metadata(self, url):
        """Fetches only the metadata header from a remote URL without downloading weights."""
        print(f"🌐 Inspecting remote LoRA: {url}")
        try:
            # Step 1: Get the size of the header (first 8 bytes)
            resp = requests.get(url, headers={**self.headers, "Range": "bytes=0-7"}, timeout=10)
            if resp.status_code not in [200, 206]:
                return {"error": f"HTTP {resp.status_code} - Link might be private or invalid."}

            header_length = struct.unpack("<Q", resp.content)[0]

            # Step 2: Fetch only the header bytes
            header_resp = requests.get(url, headers={**self.headers, "Range": f"bytes=8-{7+header_length}"})
            header_data = header_resp.json()

            metadata = header_data.get("__metadata__", {})
            return self.format_output("Remote File", metadata)
        except Exception as e:
            return {"error": f"Failed to parse remote metadata: {str(e)}"}

    def format_output(self, filename, meta):
        """Structured extraction of 'Black Box' data."""
        return {
            "File": filename,
            "Trigger": self.get_trigger_word(meta),
            "Base Model": meta.get("ss_sd_model_name", "Unknown"),
            "Resolution": f"{meta.get('ss_max_train_resolution', '???')}",
            "Dim/Rank": meta.get("ss_network_dim", "Unknown"),
            "Format": "ModelSpec" if "modelspec.title" in meta else "Kohya/Standard"
        }

    def scan_local(self, path="/content/"):
        files = glob.glob(os.path.join(path, "**/*.safetensors"), recursive=True)
        results = []
        for f in files:
            try:
                with safe_open(f, framework="pt") as lora:
                    results.append(self.format_output(os.path.basename(f), lora.metadata() or {}))
            except: continue
        return results

def run_omni_inspector(url_input=None):
    inspector = LoraOmniInspector()
    results = []

    if url_input and url_input.strip().startswith("http"):
        # Handle remote URL
        results.append(inspector.parse_remote_metadata(url_input))
    else:
        # Fallback to local scan
        print("📂 No URL provided. Scanning temporary storage (/content/)...")
        results = inspector.scan_local()

    # Professional Printout
    print("\n" + "="*85)
    print(f"{'FILE/SOURCE':<25} | {'TRIGGER':<15} | {'BASE':<10} | {'RES':<10} | {'DIM'}")
    print("="*85)
    for r in results:
        if "error" in r:
            print(f"❌ {r['error']}")
        else:
            print(f"{r['File'][:23]:<25} | {r['Trigger'][:13]:<15} | {r['Base Model'][:8]:<10} | {r['Resolution']:<10} | {r['Dim/Rank']}")
    print("="*85)

# --- USER INPUT SECTION ---
# Always ensure the URL is inside " " marks
my_link = "https://huggingface.co/user/repo/resolve/main/Appoline-05.safetensors"

if __name__ == "__main__":
    run_omni_inspector(my_link)

SyntaxError: expected 'except' or 'finally' block (348251963.py, line 28)

# Basic inspector module (First edition)

In [ ]:
from safetensors.torch import safe_open

# Load the LoRA file
with safe_open("/content/HNMF2F4S3XEAWMDN1YHNBB4HC0.safetensors", framework="pt") as f:
    metadata = f.metadata()
    if metadata:
        print(metadata)
    else:
        print("No metadata found.")